In [32]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser" #my username
password = "SNHU1234" #current password

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'], inplace=True, errors='ignore')

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))


logo = html.Img(
    src='data:image/png;base64,{}'.format(encoded_image.decode()),
    style={'height':'150px'},
    id='grazioso-logo'
)

signature = html.P("Dashboard created by Emerald Tresch — 2/22/2026")

app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    
    html.Div(
        logo,
        style={'textAlign': 'center'}
    ),

    signature,
    html.Hr(),

    html.Div([
        html.Label("Filter Animal Type:"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain Rescue', 'value': 'Mountain'},
                {'label': 'Disaster Rescue', 'value': 'Disaster'},
                {'label': 'All Animals', 'value': 'All'}
            ],
            value='All',
            inline=True
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        row_selectable="single", # Selects a single row
        page_action="native",  # Enables pagination
        page_current=0,  
        page_size=10, # Limits rows displayed per page
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),
        html.Div(id='map-id', className='col s12 m6')
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id','data'),
    [Input('filter-type', 'value')]
)

def update_dashboard(filter_type):
    if filter_type == "Water":
        query = {"breed": {"$regex": "Water"}}
    elif filter_type == "Mountain":
        query = {"breed": {"$regex": "Mountain"}}
    elif filter_type == "Disaster":
        query = {"breed": {"$regex": "Disaster"}}
    else:
        query = {}

    filtered_data = pd.DataFrame(list(db.read(query)))
    filtered_data.drop(columns=['_id'], inplace=True, errors='ignore')

    return filtered_data.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table

@app.callback(Output('graph-id', "children"),
              [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if 'breed' not in dff.columns:
        return []

    return [
        dcc.Graph(
            figure = px.pie(
                dff,
                names='breed',
                title='Preferred Animals'
            )
        )
    ]

#This callback will highlight a cell on the data table when the user selects it

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []   # No columns selected yet → return empty styling list

    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]



# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):  
    
# Safety checks
    if not viewData:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # Because we only allow single row selection, the list can be converted to a row index here
    # If no row is selected → return empty (deselect behavior)
    if not index:
        row = 0
    else:
        row = index[0]

    # Extra safety: ensure row index is valid
    if row >= len(dff):
        return []

    # Build map

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[
                        dff.iloc[row]["location_lat"],
                        dff.iloc[row]["location_long"]
                    ],
                    children=[
                        dl.Tooltip(dff.iloc[row]["breed"]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row]["name"])
                        ])
                    ]
                )
            ]
        )
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(port=9090) 

Dash app running on https://emptyquota-includebicycle-3000.codio.io/proxy/9090/
